# Multi-Weighting Matrix Product

**Data structures:**

| Set | Shape | Meaning |
|-----|-------|---------|
| `A1, A2, …` | `(n_countries,)` | one value per country |
| `W1, W2, …` | `(k,)` — same for all Wi | weight profile; `k` = number of A arrays; `Wi[j]` = weight for `Aj` in profile `i` |
| `F_a, F_b, …` | `(n_countries,)` | Y/N flag selecting which weight profile to apply |

**Goal:** for each country select one weight profile based on its flag, then compute the dot product with the stacked A values:

$$S_i = \sum_j A_j[i] \cdot W_{\text{selected}(i)}[j]$$

In [1]:
import numpy as np
import pandas as pd

# ── Country index ─────────────────────────────────────────────────────────────
countries = ['France', 'Germany', 'Italy', 'Spain', 'Poland', 'Netherlands']
n = len(countries)

# ── A arrays: one value per country  (shape: n_countries) ────────────────────
A1 = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0, 60.0], index=countries, name='A1')
A2 = pd.Series([ 5.0, 15.0, 25.0, 35.0, 45.0, 55.0], index=countries, name='A2')

# ── W arrays: weight profiles  (shape: k = 2, one element per A array) ───────
# W1[j] = weight for Aj when profile 1 (F_a = 'Y') is selected
# W2[j] = weight for Aj when profile 2 (F_a = 'N') is selected
W1 = np.array([0.7, 0.3])   # sums to 1 — a proper weighted average profile
W2 = np.array([0.2, 0.8])

# ── Flag: selects which W profile to use per country ─────────────────────────
F_a = pd.Series(['Y', 'N', 'Y', 'N', 'Y', 'N'], index=countries, name='F_a')

pd.DataFrame({'A1': A1, 'A2': A2, 'F_a': F_a}).assign(
    W1=str(W1.tolist()), W2=str(W2.tolist())
)

,A1,A2,F_a,W1,W2
France,10.0,5.0,Y,"[0.7, 0.3]","[0.2, 0.8]"
Germany,20.0,15.0,N,"[0.7, 0.3]","[0.2, 0.8]"
Italy,30.0,25.0,Y,"[0.7, 0.3]","[0.2, 0.8]"
Spain,40.0,35.0,N,"[0.7, 0.3]","[0.2, 0.8]"
Poland,50.0,45.0,Y,"[0.7, 0.3]","[0.2, 0.8]"
Netherlands,60.0,55.0,N,"[0.7, 0.3]","[0.2, 0.8]"


## Step 1 — Direct computation (reference)

For each country apply the profile selected by `F_a` and dot-product with `[A1, A2]`:

$$S[i] = \begin{cases} A1[i]\cdot W1[0] + A2[i]\cdot W1[1] & \text{if } F_a[i]=Y \\ A1[i]\cdot W2[0] + A2[i]\cdot W2[1] & \text{if } F_a[i]=N \end{cases}$$

In [2]:
flag_Y = (F_a == 'Y').astype(float)
flag_N = (F_a == 'N').astype(float)

# Apply each profile to all rows, then zero-out the inactive one
contrib_W1 = (A1 * W1[0] + A2 * W1[1]) * flag_Y   # W1 profile, active on Y-rows
contrib_W2 = (A1 * W2[0] + A2 * W2[1]) * flag_N   # W2 profile, active on N-rows

S_direct = contrib_W1 + contrib_W2
S_direct.name = 'S_direct'

pd.DataFrame({
    'F_a':       F_a,
    'contrib_W1': contrib_W1,
    'contrib_W2': contrib_W2,
    'S_direct':   S_direct,
})

,F_a,contrib_W1,contrib_W2,S_direct
France,Y,8.5,0.0,8.5
Germany,N,0.0,16.0,16.0
Italy,Y,28.5,0.0,28.5
Spain,N,0.0,36.0,36.0
Poland,Y,48.5,0.0,48.5
Netherlands,N,0.0,56.0,56.0


## Step 2 — Build the MASK matrix

**MASK** has shape `(n_countries, k)`. Each row is the weight profile selected for that country:

$$\text{MASK}[i, :] = \begin{cases} W1 & \text{if } F_a[i]=Y \\ W2 & \text{if } F_a[i]=N \end{cases}$$

Built from `W1`, `W2`, `F_a` alone — no `A` values involved.

In [3]:
# np.where broadcasts W1/W2 (length-k) across the country axis
MASK = pd.DataFrame(
    np.where((F_a == 'Y').values[:, None], W1, W2),
    index=countries,
    columns=['w_A1', 'w_A2'],
)
print("MASK  (n_countries x k):")
MASK

MASK  (n_countries x k):


,w_A1,w_A2
France,0.7,0.3
Germany,0.2,0.8
Italy,0.7,0.3
Spain,0.2,0.8
Poland,0.7,0.3
Netherlands,0.2,0.8


## Step 3 — Matrix product form  `S = rowsum(A_stack ⊙ MASK)`

Stack the A arrays column-wise into `A_stack` (same shape as `MASK`),
multiply element-wise, then sum across columns:

$$S = \mathbf{1}^\top \cdot (A_{\text{stack}} \odot \text{MASK}) \quad \text{(row-wise sum)}$$

In [4]:
A_stack = pd.DataFrame({'A1': A1, 'A2': A2})   # (n_countries x k)

S_matrix = (A_stack * MASK.values).sum(axis=1)
S_matrix.name = 'S_matrix'

assert np.allclose(S_direct, S_matrix), "Results do not match!"
print("S_matrix matches S_direct")

pd.DataFrame({
    'F_a':               F_a,
    'A_stack*MASK [A1]': (A_stack * MASK.values).iloc[:, 0],
    'A_stack*MASK [A2]': (A_stack * MASK.values).iloc[:, 1],
    'S_matrix':          S_matrix,
    'S_direct':          S_direct,
})

S_matrix matches S_direct


,F_a,A_stack*MASK [A1],A_stack*MASK [A2],S_matrix,S_direct
France,Y,7.0,1.5,8.5,8.5
Germany,N,4.0,12.0,16.0,16.0
Italy,Y,21.0,7.5,28.5,28.5
Spain,N,8.0,28.0,36.0,36.0
Poland,Y,35.0,13.5,48.5,48.5
Netherlands,N,12.0,44.0,56.0,56.0


## Summary

```
MASK[i, :] = W1  if F_a[i]='Y'      ← np.where broadcast
             W2  if F_a[i]='N'

S = rowsum( A_stack ⊙ MASK )         ← (A_stack * MASK).sum(axis=1)
```

Scales to **k** arrays and **any number of flags** by adding columns to `A_stack` / `MASK`
and extending the `np.where` selection logic.

## Best answer — clean self-contained snippet

In [5]:
import numpy as np
import pandas as pd

# ── Inputs ────────────────────────────────────────────────────────────────────
countries = ['France', 'Germany', 'Italy', 'Spain', 'Poland', 'Netherlands']

A1  = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0, 60.0], index=countries)
A2  = pd.Series([ 5.0, 15.0, 25.0, 35.0, 45.0, 55.0], index=countries)

W1  = np.array([0.7, 0.3])   # weight profile used when F_a = 'Y'
W2  = np.array([0.2, 0.8])   # weight profile used when F_a = 'N'

F_a = pd.Series(['Y', 'N', 'Y', 'N', 'Y', 'N'], index=countries)

# ── Build MASK  (n_countries x k) from W profiles and flag alone ──────────────
MASK = np.where((F_a == 'Y').values[:, None], W1, W2)   # broadcast W1 / W2 row-wise

# ── Stack A arrays to match MASK shape ───────────────────────────────────────
A_stack = np.column_stack([A1, A2])                      # (n_countries x k)

# ── S = rowsum( A_stack ⊙ MASK ) ─────────────────────────────────────────────
S = pd.Series((A_stack * MASK).sum(axis=1), index=countries, name='S')

S


France          8.5
Germany        16.0
Italy          28.5
Spain          36.0
Poland         48.5
Netherlands    56.0
Name: S, dtype: float64

---

# Table → Array pipelines

Each source table `Ti` has **countries as rows** and **years as columns**.  
An array `Ai` is produced by chaining one or more operations on `Ti`:

```
Ti  →  op_1  →  op_2  →  …  →  Ai
```

**Design:** every operation is a *factory* — a function that accepts configuration
parameters and returns a callable with signature

```
op(DataFrame | Series) → DataFrame | Series
```

A `make_pipeline(*ops)` combinator chains them left-to-right via `functools.reduce`,
mirroring the delegate/chain-of-responsibility pattern.

In [ ]:
import numpy as np
import pandas as pd

# ── Sample tables: rows = countries, columns = years ─────────────────────────
rng = np.random.default_rng(42)

countries = ['France', 'Germany', 'Italy', 'Spain', 'Poland', 'Netherlands']
years     = list(range(2018, 2024))   # 2018 … 2023

T1 = pd.DataFrame(
    rng.uniform(50, 150, size=(len(countries), len(years))),
    index=countries, columns=years,
)
T2 = pd.DataFrame(
    rng.uniform(1, 10, size=(len(countries), len(years))),
    index=countries, columns=years,
)

print("T1"); display(T1.round(2))
print("T2"); display(T2.round(2))

In [ ]:
from functools import reduce

# ══════════════════════════════════════════════════════════════════════════════
# Atomic operation factories
# Each returns a callable:  (DataFrame | Series) → (DataFrame | Series)
# ══════════════════════════════════════════════════════════════════════════════

def op_weighted_avg(k, weights=None):
    """Weighted average over the last k year-columns.
    Matrix form: (n, k) @ (k,) → (n,).   DataFrame → Series."""
    def _apply(df):
        block = df.iloc[:, -k:].values              # (n_countries, k)
        w = np.ones(k) if weights is None else np.asarray(weights, float)
        w = w / w.sum()                             # normalise to sum=1
        return pd.Series(block @ w, index=df.index)
    return _apply


def op_moving_avg(window):
    """Row-wise rolling mean over year-columns.
    Matrix form: (n, p) @ (p, p-w+1) → (n, p-w+1).   DataFrame → DataFrame."""
    def _apply(df):
        vals  = df.values                           # (n_countries, n_years)
        p     = vals.shape[1]
        n_out = p - window + 1
        # Averaging kernel matrix: each row selects a window of columns
        K = np.zeros((n_out, p))
        for i in range(n_out):
            K[i, i : i + window] = 1.0 / window    # (n_out, p)
        result = vals @ K.T                         # (n, p) @ (p, n_out) → (n, n_out)
        new_cols = df.columns[window - 1 :]
        return pd.DataFrame(result, index=df.index, columns=new_cols)
    return _apply


def op_row_mean():
    """Plain mean over all year-columns.
    Matrix form: (n, p) @ (p,) → (n,).   DataFrame → Series."""
    def _apply(df):
        p = df.shape[1]
        w = np.ones(p) / p
        return pd.Series(df.values @ w, index=df.index)
    return _apply


def op_select_year(year):
    """Extract a single year column.   DataFrame → Series."""
    return lambda df: df[year]


def op_log():
    """Natural log, element-wise.   DataFrame | Series → same shape."""
    return np.log


def op_sqrt():
    """Square root, element-wise.   DataFrame | Series → same shape."""
    return np.sqrt

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Pipeline combinator
# ══════════════════════════════════════════════════════════════════════════════

def make_pipeline(*ops):
    """Chain callables left-to-right.
    Returns a single callable: data → ops[-1]( … ops[1]( ops[0](data) ) … )"""
    def _run(data):
        return reduce(lambda x, f: f(x), ops, data)
    return _run

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Define chains and apply them
#
#   T1  →  weighted_avg(last 3 yrs, weights=[0.2,0.3,0.5])  →  sqrt  →  A1
#   T2  →  select_year(2021)  →  log  →  A2
#   T2  →  moving_avg(window=3)  →  row_mean  →  A3
# ══════════════════════════════════════════════════════════════════════════════

chain_A1 = make_pipeline(
    op_weighted_avg(k=3, weights=[0.2, 0.3, 0.5]),   # (n,6)→(n,)
    op_sqrt(),                                         # (n,)→(n,)
)

chain_A2 = make_pipeline(
    op_select_year(2021),                              # (n,6)→(n,)
    op_log(),                                          # (n,)→(n,)
)

chain_A3 = make_pipeline(
    op_moving_avg(window=3),                           # (n,6)→(n,4)
    op_row_mean(),                                     # (n,4)→(n,)
)

A1 = chain_A1(T1).rename('A1')
A2 = chain_A2(T2).rename('A2')
A3 = chain_A3(T2).rename('A3')

pd.DataFrame({'A1': A1, 'A2': A2, 'A3': A3}).round(4)